In [1]:
# 1 - imports and Architecture

import cv2
import os
import numpy as np
import time
import psutil
import math
from datetime import datetime
import mediapipe as mp  

print("\n" + "="*50)
print("EDGE AI SMART CLASSROOM - SYSTEM ARCHITECTURE")
print("="*50)
print("- Recognition Model : OpenCV LBPH (Texture-based, 0 parameters)")
print("- Liveness Model    : MediaPipe Face Mesh (TFLite CNN, < 2MB, ~468 landmarks)")
print("- Edge Optimization : Grayscale conversion, 200x200 ROI cropping")
print("- Hardware Target   : CPU-only inference (Local Processing)")
print("="*50 + "\n")


EDGE AI SMART CLASSROOM - SYSTEM ARCHITECTURE
- Recognition Model : OpenCV LBPH (Texture-based, 0 parameters)
- Liveness Model    : MediaPipe Face Mesh (TFLite CNN, < 2MB, ~468 landmarks)
- Edge Optimization : Grayscale conversion, 200x200 ROI cropping
- Hardware Target   : CPU-only inference (Local Processing)



In [2]:
# 2 - Training the offline face recognizer (FIXED WITH MEDIAPIPE CROPPING)

import cv2
import os
import numpy as np
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

KNOWN_FACES_DIR = "known_faces"
os.makedirs(KNOWN_FACES_DIR, exist_ok=True)

recognizer = cv2.face.LBPHFaceRecognizer_create()
label_map = {}
faces, labels = [], []

# Use MediaPipe to crop the training photos exactly how the live camera does
base_options = python.BaseOptions(model_asset_path='face_landmarker.task')
options = vision.FaceLandmarkerOptions(base_options=base_options, num_faces=1)
training_landmarker = vision.FaceLandmarker.create_from_options(options)

print("[INFO] Training LBPH model on tight MediaPipe crops...")
current_id = 0

for filename in os.listdir(KNOWN_FACES_DIR):
    if filename.lower().endswith(('.jpg', '.jpeg', '.png')):
        name = os.path.splitext(filename)[0].split('_')[0].capitalize()
        
        if name not in label_map.values():
            label_map[current_id] = name
            person_id = current_id
            current_id += 1
        else:
            person_id = [k for k, v in label_map.items() if v == name][0]

        img_path = os.path.join(KNOWN_FACES_DIR, filename)
        frame = cv2.imread(img_path)
        if frame is None: continue
        
        # Process image with MediaPipe
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
        detection_result = training_landmarker.detect(mp_image)
        
        if detection_result.face_landmarks:
            img_h, img_w, _ = frame.shape
            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            face_landmarks = detection_result.face_landmarks[0]
            
            # Exact same crop coordinates as live camera
            x_coords = [int(lm.x * img_w) for lm in face_landmarks]
            y_coords = [int(lm.y * img_h) for lm in face_landmarks]
            
            x_min, x_max = max(0, min(x_coords)), min(img_w, max(x_coords))
            y_min, y_max = max(0, min(y_coords)), min(img_h, max(y_coords))
            
            padding = int((y_max - y_min) * 0.1)
            y_min, y_max = max(0, y_min - padding * 2), min(img_h, y_max + padding)
            x_min, x_max = max(0, x_min - padding), min(img_w, x_max + padding)
            
            face_roi = gray[y_min:y_max, x_min:x_max]
            if face_roi.shape[0] < 50 or face_roi.shape[1] < 50: continue
            
            face_roi_resized = cv2.resize(face_roi, (200, 200))
            faces.append(face_roi_resized)
            labels.append(person_id)

if len(faces) > 0:
    recognizer.train(faces, np.array(labels))
    print(f"[INFO] Training complete. Registered students: {list(label_map.values())}")
else:
    print("[WARN] No faces detected in training images by MediaPipe.")

[INFO] Training LBPH model on tight MediaPipe crops...
[INFO] Training complete. Registered students: ['Ibrahim']


In [3]:
#3 - Mediapipe and Attendance logic

import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

# Initialize the modern MediaPipe Face Landmarker Task
base_options = python.BaseOptions(model_asset_path='face_landmarker.task')
options = vision.FaceLandmarkerOptions(
    base_options=base_options,
    output_face_blendshapes=False,
    num_faces=5
)
face_landmarker = vision.FaceLandmarker.create_from_options(options)

LEFT_EYE = [33, 160, 158, 133, 153, 144]
RIGHT_EYE = [362, 385, 387, 263, 373, 380]

def calculate_ear(landmarks, eye_indices, img_w, img_h):
    pts = [(int(landmarks[idx].x * img_w), int(landmarks[idx].y * img_h)) for idx in eye_indices]
    d_v1 = math.dist(pts[1], pts[5])
    d_v2 = math.dist(pts[2], pts[4])
    d_h = math.dist(pts[0], pts[3])
    if d_h == 0: return 0.3
    return (d_v1 + d_v2) / (2.0 * d_h)

def get_today_filename():
    return f"attendance_{datetime.now().strftime('%Y-%m-%d')}.csv"

verified_students = set()
csv_file = get_today_filename()
if os.path.isfile(csv_file):
    with open(csv_file, 'r') as f:
        for line in f.readlines()[1:]:
            verified_students.add(line.split(',')[0].strip())

def mark_attendance(name):
    filename = get_today_filename()
    if not os.path.isfile(filename):
        with open(filename, 'w') as f: f.write("Name,Time\n")
            
    if name not in verified_students:
        time_str = datetime.now().strftime('%H:%M:%S')
        with open(filename, 'a') as f: f.write(f"{name},{time_str}\n")
        verified_students.add(name)
        print(f"[LOG] Attendance recorded for {name} at {time_str}")

recently_marked_timers = {} 
blink_counters = {}

In [4]:
# 4 - Live Camera Loop (INSTANT ATTENDANCE + 3-SECOND FLICKER LOCK)

cap = cv2.VideoCapture(0)
print("[INFO] Starting Edge AI feed... Press 'q' to exit.")

prev_time = time.time()

# This dictionary acts as our 3-second memory buffer to prevent flickering
face_memory = {}

try:
    while True:
        start_time = time.time()
        ret, frame = cap.read()
        if not ret: break

        frame = cv2.flip(frame, 1) 
        img_h, img_w, _ = frame.shape
        
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        gray_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        
        # Convert to MediaPipe Image format
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
        detection_result = face_landmarker.detect(mp_image)

        if detection_result.face_landmarks and len(faces) > 0:
            for face_landmarks in detection_result.face_landmarks:
                # Extract coordinates
                x_coords = [int(lm.x * img_w) for lm in face_landmarks]
                y_coords = [int(lm.y * img_h) for lm in face_landmarks]
                
                x_min, x_max = max(0, min(x_coords)), min(img_w, max(x_coords))
                y_min, y_max = max(0, min(y_coords)), min(img_h, max(y_coords))
                
                padding = int((y_max - y_min) * 0.1)
                y_min, y_max = max(0, y_min - padding * 2), min(img_h, y_max + padding)
                x_min, x_max = max(0, x_min - padding), min(img_w, x_max + padding)

                face_roi = gray_frame[y_min:y_max, x_min:x_max]
                if face_roi.shape[0] < 50 or face_roi.shape[1] < 50: continue

                face_roi_resized = cv2.resize(face_roi, (200, 200))
                
                # Predict
                label_id, confidence = recognizer.predict(face_roi_resized)
                
                name = "Unknown"
                match_acc_str = ""
                
                # Strict security threshold
                if confidence < 80:
                    name = label_map.get(label_id, "Unknown")
                    acc_pct = max(0, min(100, round(100 - (confidence / 1.5))))
                    match_acc_str = f" | Conf: {acc_pct}%"
                    
                    # Update our memory buffer with the exact time we saw you
                    if name != "Unknown":
                        face_memory[name] = time.time()
                else:
                    # FLICKER LOCK LOGIC: If we don't recognize you this frame, 
                    # check if we successfully saw you within the last 3 seconds.
                    for mem_name, last_seen in list(face_memory.items()):
                        if time.time() - last_seen < 3.0:
                            name = mem_name
                            match_acc_str = " | (Locked)"
                            break

                # Instant Attendance (No Blink Required)
                if name != "Unknown" and name not in verified_students:
                    mark_attendance(name)
                    recently_marked_timers[name] = time.time()

                # UI Box Coloring & Text State
                box_color = (0, 0, 255)  # Default Red for Unknown
                display_text = "Unknown"
                
                if name != "Unknown":
                    if name in verified_students:
                        box_color = (0, 255, 0) # Green
                        # Show the success message for 3-4 seconds
                        if name in recently_marked_timers and (time.time() - recently_marked_timers[name] < 4):
                            display_text = f"{name}{match_acc_str} - Present!"
                        else:
                            display_text = f"{name}{match_acc_str} - Already Present"

                # Draw Face Bounding Box & Status Text
                cv2.rectangle(frame, (x_min, y_min), (x_max, y_max), box_color, 2)
                (text_w, text_h), _ = cv2.getTextSize(display_text, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 2)
                cv2.rectangle(frame, (x_min, y_min - 25), (x_min + text_w, y_min), box_color, -1)
                cv2.putText(frame, display_text, (x_min, y_min - 7), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 2)

        # EDGE AI METRICS HUD (Floating Text without Black Box)
        latency_ms = (time.time() - start_time) * 1000
        fps = 1.0 / (time.time() - prev_time)
        prev_time = time.time()
        
        process = psutil.Process(os.getpid())
        memory_mb = process.memory_info().rss / (1024 * 1024)

        # Helper function to draw text with a black outline so it doesn't blend into white walls
        def draw_hud_text(img, text, pos, color):
            cv2.putText(img, text, pos, cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 3) # Outline
            cv2.putText(img, text, pos, cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 1)     # Inner Text

        draw_hud_text(frame, "EDGE AI PIPELINE", (15, 30), (255, 255, 255))
        draw_hud_text(frame, "Model: LBPH + MP Tasks", (15, 55), (255, 255, 255))
        draw_hud_text(frame, f"Latency: {latency_ms:.0f} ms", (15, 85), (255, 255, 0))
        draw_hud_text(frame, f"FPS: {fps:.1f}", (15, 110), (0, 255, 0))
        draw_hud_text(frame, f"RAM Usage: {memory_mb:.0f} MB", (15, 135), (0, 255, 255))

        cv2.imshow("EdgeFace Smart Classroom", frame)

        if cv2.waitKey(1) & 0xFF == ord('q'): 
            break

finally:
    cap.release()
    cv2.destroyAllWindows()
    print("[INFO] Camera released and windows destroyed safely.")

[INFO] Starting Edge AI feed... Press 'q' to exit.
[INFO] Camera released and windows destroyed safely.
